# Feature Engineering and Evidence Curves

Stage 04 translates stage-03 spike trains into rate and timing evidence used by the behavioral discrimination models. This notebook is meant to explain the logic, not to rerun expensive simulations.


## Pipeline Map

Stage 04 uses the archived CV0-based response table to:

1. Convert spike trains into rate features: spike count, burst count, and spikes per burst.
2. Convert spike trains into timing features: local variation (LV) of inter-spike and inter-burst intervals.
3. Convert rate features into evidence using a fixed Weber-like hard just-noticeable-difference (JND) rule.
4. Average normalized evidence across repetitions for downstream model fitting.


In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display

def _bootstrap_src(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        src_dir = candidate / 'src'
        if src_dir.exists() and (candidate / 'data').exists():
            if str(src_dir) not in sys.path:
                sys.path.insert(0, str(src_dir))
            return candidate
    raise RuntimeError('Could not locate repository root.')

REPO_ROOT = _bootstrap_src(Path.cwd())

from utils import require_data_files, data_subdir, ensure_src_on_path, find_repo_root
from figures import set_manuscript_style

REPO_ROOT = find_repo_root(REPO_ROOT)
SRC_DIR = ensure_src_on_path(REPO_ROOT)
require_data_files(
    REPO_ROOT,
    [
        'data/detection/generated/cv0_based_responses.csv',
    ],
    context='04_FeatureEngineering',
)


from discrimination import (
    DEFAULT_FREQS,
    FeatureEngineeringConfig,
    feature_metric_pivot,
    feature_metric_summary_cached,
    load_feature_simulation,
    make_feature_evidence_widget,
    prepare_feature_workflow,
)

set_manuscript_style(dpi=120)
pd.set_option('display.max_columns', None)

## Configure and Check Data

The notebook reads `data/detection/generated/cv0_based_responses.csv`. The data check is intentionally quiet unless the file is missing.


In [ ]:
detection_dir = data_subdir(REPO_ROOT, 'detection') / 'generated'
config = FeatureEngineeringConfig(
    sim_csv=detection_dir / 'cv0_based_responses.csv',
    freqs=tuple(DEFAULT_FREQS),
)
if not config.sim_csv.exists():
    raise FileNotFoundError(f'Missing stage-03 response table: {config.sim_csv}')

feature_sim = load_feature_simulation(config)
workflow = prepare_feature_workflow(feature_sim, config)

pd.DataFrame([
    {'setting': 'analysis_window_ms', 'value': f'{config.t0_ms:g}-{config.t1_ms:g}'},
    {'setting': 'bins_ms', 'value': ', '.join(f'{b:g}' for b in config.bins)},
    {'setting': 'hard_jnd_k', 'value': config.jnd_k},
    {'setting': 'rows_loaded', 'value': f'{len(feature_sim.data):,}'},
])

## Step 1: Rate Features

Rate features summarize how many axonal spikes occur and how those spikes group into 10 ms burst-like events. These tables show means across repetitions for every frequency, CV, and threshold type.


In [ ]:
rate_feature_labels = {
    'n_spikes': 'Mean n_spikes',
    'n_bursts': 'Mean n_bursts',
    'spikes_per_burst': 'Mean spikes_per_burst',
}

for metric, label in rate_feature_labels.items():
    display(Markdown(f'### {label}'))
    metric_summary = feature_metric_summary_cached(workflow, metric, freqs=config.freqs)
    display(feature_metric_pivot(metric_summary).round(3))

## Step 2: Timing Features

Timing features describe irregularity in the intervals between spikes or bursts. LV is near zero for regular intervals and increases when neighboring intervals differ strongly.


In [ ]:
timing_feature_labels = {
    'LV_ISI': 'Mean LV_ISI',
    'LV_IBI': 'Mean LV_IBI',
}

for metric, label in timing_feature_labels.items():
    display(Markdown(f'### {label}'))
    metric_summary = feature_metric_summary_cached(workflow, metric, freqs=config.freqs)
    display(feature_metric_pivot(metric_summary).round(3))

## Step 3: Interactive Feature/Evidence Walkthrough

The widget below uses the fixed manuscript JND setting (`k=2.5`) and shows how one repetition's spike train becomes rate features, timing features, raw evidence, and mean normalized evidence curves.


In [ ]:
display(make_feature_evidence_widget(workflow))

---